In [ ]:
# ===========================================================================
# PRAAT ACOUSTIC ANALYSIS - interactive driver (Phase 4)
#
# All extraction/plotting logic lives in src/; this notebook only calls it
# and stores results, matching notebooks/01 and 02's convention.
#
#   src/praat.py           per-feature-group extraction (F0, jitter, shimmer,
#                           HNR, formants, intensity, speech-rate/pause/voice-
#                           break proxies) + extract_praat_features_batch()
#   src/visualization.py   plot_praat_feature_comparison, build_praat_group_summary
#
# Features are extracted from the ORIGINAL audio, not the VAD-trimmed/
# zero-padded window the Deep/Acoustic pathways train on - jitter, shimmer,
# HNR, and formants are only meaningful on natural speech. See ROADMAP.md
# Phase 4 for the full plan and src/praat.py's module docstring for the
# speech-rate/pause-duration caveat (UA-Speech utterances are single
# isolated words, not continuous speech).
# ===========================================================================

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.console import print_header, print_kv
from src.training.data import load_manifest

config.ensure_directories()

df_m6 = load_manifest()

print_header("Praat Acoustic Analysis")
print_kv("Manifest", config.MANIFEST_PATH)
print_kv("Utterances", len(df_m6))

In [ ]:
# STAGE 1 - Extract all 30 Praat features for every utterance in the manifest:
#   pitch        f0 mean/max/min/std/range        (std+range = monopitch)
#   perturbation jitter x4, shimmer x4            (vocal-fold instability)
#   noise        hnr mean/std/min                 (breathiness, roughness)
#   articulation f1/f2/f3 mean+std, f2_f1_ratio   (vowel-space centralization)
#   loudness     intensity mean/max/min/std       (loudness control)
#   rhythm       speech_rate, pause_duration, voice_breaks
#
# Cached to outputs/praat_features.csv - safe to re-run this cell, later runs
# load the cache instead of recomputing ~21k files (~25-30 minutes uncached).
#
# NOTE: a cache written by the earlier 18-feature version of src/praat.py is
# detected as stale (missing columns) and automatically re-extracted, so this
# cell must be re-run once to pick up the 12 added features.
from src.praat import extract_praat_features_batch

praat_features = extract_praat_features_batch(df_m6, cache_path=config.PRAAT_FEATURES_PATH)
praat_features.head()

In [ ]:
# STAGE 2 - Compare Healthy vs Very Low vs Low vs Mid vs High severity groups:
# box-plot grid for all 30 features, saved to outputs/figures/, plus a
# group-means +/- std table saved to outputs/metrics/.
from src.visualization import plot_praat_feature_comparison, build_praat_group_summary

figure_path = plot_praat_feature_comparison(praat_features, show=True)
group_summary = build_praat_group_summary(praat_features)

summary_path = config.METRICS_DIR / "praat_severity_group_summary.csv"
group_summary.to_csv(summary_path)

print_header("Phase 4 - Severity Group Comparison")
print_kv("Figure", figure_path)
print_kv("Group summary table", summary_path)
group_summary

In [ ]:
# STAGE 3 - Which of those group differences are actually real?
#
# The box plots above are descriptive; this is the test. Kruskal-Wallis H per
# feature across the five groups (non-parametric, because jitter/shimmer and the
# pause/voice-break proxies are bounded and heavily skewed, so ANOVA's normality
# assumption does not hold), Bonferroni-corrected across the 30 features so that
# testing them all at once does not manufacture significance.
#
# The significant rows are the features the discussion section can legitimately
# claim separate severity levels - and they are the ones worth reading first in
# Phase 5's error analysis and Phase 6's Praat fusion pathway.
from src.praat import praat_group_significance

significance = praat_group_significance(praat_features)

significance_path = config.METRICS_DIR / "praat_significance.csv"
significance.to_csv(significance_path, index=False)

n_sig = int(significance["significant"].sum())

print_header("Phase 4 - Kruskal-Wallis Severity Group Significance")
print_kv("Significance table", significance_path)
print_kv("Significant features (p_adj < 0.05)", f"{n_sig} / {len(significance)}")
print_kv("Strongest separator", significance.iloc[0]["feature"])
significance